# ALS Baseline: Books + Movies Intersection

Train one implicit ALS model on merged Books and Movies interactions from the prepared Google Drive splits. Recommendations are filtered back to the target domain before MAP@5/NDCG@10/Recall@10 evaluation.

In [ ]:
!test -d /content/recommendation-system/.git \
  || git clone -b als https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin als
!git checkout als
!git reset --hard origin/als
!git log -1 --oneline
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import implicit
import numpy as np
import pandas as pd

from configs.model_configs import ALS_CONFIG
from src.data.build_matrix import add_domain_item_ids, build_implicit_als_matrix
from src.evaluation.metrics import map_at_k, ndcg_at_k, recall_at_k
from src.evaluation.cross_domain_eval import evaluate_multi_domain

## Load Prepared Splits

The split CSVs are expected to contain the Books/Movies intersecting users and time-based train/validation/test rows.

In [ ]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [ ]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

## Shared Indexes and Train Matrix

In [ ]:
interaction_matrix = build_implicit_als_matrix(train_df)

user_item_train = interaction_matrix.user_item_train
user2idx = interaction_matrix.user2idx
item2idx = interaction_matrix.item2idx
idx2item = interaction_matrix.idx2item
item_domain = interaction_matrix.item_domain
train_seen_idx_by_user = interaction_matrix.train_seen_idx_by_user
domain_item_indices = interaction_matrix.domain_item_indices

n_users, n_items = user_item_train.shape

print(f'users={n_users:,}, items={n_items:,}, train_interactions={user_item_train.nnz:,}')
print(train_df['domain'].value_counts())

In [ ]:
assert user_item_train.shape == (n_users, n_items)
assert len(domain_item_indices['books']) > 0
assert len(domain_item_indices['movies']) > 0

## Train Collective ALS

In [ ]:
FACTORS = ALS_CONFIG["factors"]
REGULARIZATION = ALS_CONFIG["regularization"]
ITERATIONS = ALS_CONFIG["iterations"]
ALPHA = ALS_CONFIG["alpha"]
K = ALS_CONFIG["k"]
RANDOM_STATE = ALS_CONFIG["random_state"]

als_model = implicit.als.AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=RANDOM_STATE,
)

item_user_train = (user_item_train * ALPHA).T.tocsr()
als_model.fit(item_user_train)

# The model is fit on item-user input, so implicit stores item vectors in
# user_factors and user vectors in item_factors.
als_item_factors = als_model.user_factors
als_user_factors = als_model.item_factors

assert als_item_factors.shape[0] == n_items
assert als_user_factors.shape[0] == n_users

## Domain-Filtered Recommendations

In [ ]:
_candidate_to_position = {
    domain: {item_idx: pos for pos, item_idx in enumerate(indices)}
    for domain, indices in domain_item_indices.items()
}

def relevant_items_by_user(split_df, target_domain):
    domain_rows = split_df[split_df['domain'] == target_domain]
    return domain_rows.groupby('user_id')['item_id'].agg(set).to_dict()


def recommend_for_users(user_ids, target_domain, k=10, batch_size=512, max_score_elements=20_000_000):
    candidates = domain_item_indices[target_domain]
    candidate_factors = als_item_factors[candidates]
    candidate_to_position = _candidate_to_position[target_domain]  # O(1) instant reuse

    recommendations = {}
    user_ids = list(user_ids)
    effective_batch_size = max(1, min(batch_size, max_score_elements // max(len(candidates), 1)))

    for batch_start in range(0, len(user_ids), effective_batch_size):
        batch = user_ids[batch_start:batch_start + effective_batch_size]
        valid_user_ids = []
        user_indices = []

        for user_id in batch:
            user_idx = user2idx.get(user_id)
            if user_idx is None:
                recommendations[user_id] = []
                continue
            valid_user_ids.append(user_id)
            user_indices.append(user_idx)

        if not user_indices:
            continue

        user_factors = als_user_factors[np.array(user_indices, dtype=np.int64)]

        # Compute scores for the entire batch instantly
        all_scores = user_factors @ candidate_factors.T

        # VECTORIZED MASKING: Extract history blocks for the whole batch at once
        for idx, user_idx in enumerate(user_indices):
            seen_items = train_seen_idx_by_user.get(user_idx, set())
            # Fast list comprehension via local O(1) dictionary lookups
            known_pos = [candidate_to_position[i] for i in seen_items if i in candidate_to_position]
            if known_pos:
                all_scores[idx, known_pos] = -np.inf

        # Run argpartition on the fully masked matrix in parallel
        for row_idx, user_id in enumerate(valid_user_ids):
            scores = all_scores[row_idx]
            finite_count = np.isfinite(scores).sum()
            if finite_count == 0:
                recommendations[user_id] = []
                continue

            top_n = min(k, int(finite_count))
            top_positions = np.argpartition(-scores, top_n - 1)[:top_n]
            top_positions = top_positions[np.argsort(-scores[top_positions])]
            recommendations[user_id] = [idx2item[int(candidates[pos])] for pos in top_positions]

        if (batch_start // effective_batch_size) % 20 == 0:
            print(f'  Processed {batch_start:,}/{len(user_ids):,} users for {target_domain}...')

    return recommendations

## Evaluate and Save Metrics

In [ ]:
K = ALS_CONFIG["k"]

# 1. Clear interface router mapping to notebook inference algorithm
def run_als_inference(user_ids, target_domain, k=10):
    return recommend_for_users(user_ids, target_domain, k=k)

# 2. Unified, safe execution sweep across splits simultaneously
valid_results, test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=run_als_inference,
    k=K,
    map_k=5,
)

# 3. Correctly label model payload before logging to CSV
os.makedirs('results', exist_ok=True)
result_row = {
    'model': 'implicit_als_books_movies_joint_weighted',  # Corrected name label
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': n_users,
    'n_items': n_items,
    'n_train_interactions': user_item_train.nnz,
    'books_valid_ndcg@10': valid_results['books']['ndcg@10'],
    'movies_valid_ndcg@10': valid_results['movies']['ndcg@10'],
    'valid_macro_ndcg@10': round((valid_results['books']['ndcg@10'] + valid_results['movies']['ndcg@10']) / 2, 4),
    'books_valid_recall@10': valid_results['books']['recall@10'],
    'movies_valid_recall@10': valid_results['movies']['recall@10'],
    'valid_macro_recall@10': round((valid_results['books']['recall@10'] + valid_results['movies']['recall@10']) / 2, 4),
    'books_valid_map@5': valid_results['books']['map@5'],
    'movies_valid_map@5': valid_results['movies']['map@5'],
    'valid_macro_map@5': round((valid_results['books']['map@5'] + valid_results['movies']['map@5']) / 2, 4),
    'books_test_ndcg@10': test_results['books']['ndcg@10'],
    'movies_test_ndcg@10': test_results['movies']['ndcg@10'],
    'test_macro_ndcg@10': round((test_results['books']['ndcg@10'] + test_results['movies']['ndcg@10']) / 2, 4),
    'books_test_recall@10': test_results['books']['recall@10'],
    'movies_test_recall@10': test_results['movies']['recall@10'],
    'test_macro_recall@10': round((test_results['books']['recall@10'] + test_results['movies']['recall@10']) / 2, 4),
    'books_test_map@5': test_results['books']['map@5'],
    'movies_test_map@5': test_results['movies']['map@5'],
    'test_macro_map@5': round((test_results['books']['map@5'] + test_results['movies']['map@5']) / 2, 4),
}

results_df = pd.DataFrame([result_row])
results_df.to_csv('results/als_baseline_metrics.csv', index=False)
results_df

## Ablation: Single-Domain ALS

Train separate Books-only and Movies-only ALS models with the same hyperparameters as the collective model, then compare in-domain ranking performance against the collective ALS baseline.

In [ ]:
def train_single_domain_als(domain_train_df):
    domain_matrix = build_implicit_als_matrix(domain_train_df)
    domain_user_item_train = domain_matrix.user_item_train

    domain_model = implicit.als.AlternatingLeastSquares(
        factors=FACTORS,
        regularization=REGULARIZATION,
        iterations=ITERATIONS,
        random_state=RANDOM_STATE,
    )

    domain_item_user_train = (domain_user_item_train * ALPHA).T.tocsr()
    domain_model.fit(domain_item_user_train)

    return {
        'model': domain_model,
        'matrix': domain_matrix,
        'user_item_train': domain_user_item_train,
        'item_factors': domain_model.user_factors,
        'user_factors': domain_model.item_factors,
        'candidate_to_position': {
            domain: {item_idx: pos for pos, item_idx in enumerate(indices)}
            for domain, indices in domain_matrix.domain_item_indices.items()
        },
    }


def recommend_single_domain_als(
    user_ids,
    target_domain,
    *,
    trained_domain_model,
    k=10,
    batch_size=512,
    max_score_elements=20_000_000,
):
    domain_matrix = trained_domain_model['matrix']
    candidates = domain_matrix.domain_item_indices[target_domain]
    candidate_factors = trained_domain_model['item_factors'][candidates]
    candidate_to_position = trained_domain_model['candidate_to_position'][target_domain]

    recommendations = {}
    user_ids = list(user_ids)
    effective_batch_size = max(1, min(batch_size, max_score_elements // max(len(candidates), 1)))

    for batch_start in range(0, len(user_ids), effective_batch_size):
        batch = user_ids[batch_start:batch_start + effective_batch_size]
        valid_user_ids = []
        user_indices = []

        for user_id in batch:
            user_idx = domain_matrix.user2idx.get(user_id)
            if user_idx is None:
                recommendations[user_id] = []
                continue
            valid_user_ids.append(user_id)
            user_indices.append(user_idx)

        if not user_indices:
            continue

        user_factors = trained_domain_model['user_factors'][np.array(user_indices, dtype=np.int64)]
        all_scores = user_factors @ candidate_factors.T

        for idx, user_idx in enumerate(user_indices):
            seen_items = domain_matrix.train_seen_idx_by_user.get(user_idx, set())
            known_pos = [candidate_to_position[i] for i in seen_items if i in candidate_to_position]
            if known_pos:
                all_scores[idx, known_pos] = -np.inf

        for row_idx, user_id in enumerate(valid_user_ids):
            scores = all_scores[row_idx]
            finite_count = np.isfinite(scores).sum()
            if finite_count == 0:
                recommendations[user_id] = []
                continue

            top_n = min(k, int(finite_count))
            top_positions = np.argpartition(-scores, top_n - 1)[:top_n]
            top_positions = top_positions[np.argsort(-scores[top_positions])]
            recommendations[user_id] = [domain_matrix.idx2item[int(candidates[pos])] for pos in top_positions]

        if (batch_start // effective_batch_size) % 20 == 0:
            print(f'  Processed {batch_start:,}/{len(user_ids):,} users for {target_domain}...')

    return recommendations


In [ ]:
books_only_model = train_single_domain_als(books_train)

def run_books_only_als_inference(user_ids, target_domain, k=10):
    return recommend_single_domain_als(
        user_ids,
        target_domain,
        trained_domain_model=books_only_model,
        k=k,
    )

books_only_valid_results, books_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'books'],
        'test': test_df[test_df['domain'] == 'books'],
    },
    recommend_func=run_books_only_als_inference,
    k=K,
    map_k=5,
)

books_only_result_row = {
    'model': 'implicit_als_books_only_weighted',
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': books_only_model['user_item_train'].shape[0],
    'n_items': books_only_model['user_item_train'].shape[1],
    'n_train_interactions': books_only_model['user_item_train'].nnz,
    'books_valid_ndcg@10': books_only_valid_results['books']['ndcg@10'],
    'movies_valid_ndcg@10': np.nan,
    'valid_macro_ndcg@10': books_only_valid_results['books']['ndcg@10'],
    'books_valid_recall@10': books_only_valid_results['books']['recall@10'],
    'movies_valid_recall@10': np.nan,
    'valid_macro_recall@10': books_only_valid_results['books']['recall@10'],
    'books_valid_map@5': books_only_valid_results['books']['map@5'],
    'movies_valid_map@5': np.nan,
    'valid_macro_map@5': books_only_valid_results['books']['map@5'],
    'books_test_ndcg@10': books_only_test_results['books']['ndcg@10'],
    'movies_test_ndcg@10': np.nan,
    'test_macro_ndcg@10': books_only_test_results['books']['ndcg@10'],
    'books_test_recall@10': books_only_test_results['books']['recall@10'],
    'movies_test_recall@10': np.nan,
    'test_macro_recall@10': books_only_test_results['books']['recall@10'],
    'books_test_map@5': books_only_test_results['books']['map@5'],
    'movies_test_map@5': np.nan,
    'test_macro_map@5': books_only_test_results['books']['map@5'],
}

books_only_result_row


In [ ]:
movies_only_model = train_single_domain_als(movies_train)

def run_movies_only_als_inference(user_ids, target_domain, k=10):
    return recommend_single_domain_als(
        user_ids,
        target_domain,
        trained_domain_model=movies_only_model,
        k=k,
    )

movies_only_valid_results, movies_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'movies'],
        'test': test_df[test_df['domain'] == 'movies'],
    },
    recommend_func=run_movies_only_als_inference,
    k=K,
    map_k=5,
)

movies_only_result_row = {
    'model': 'implicit_als_movies_only_weighted',
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': movies_only_model['user_item_train'].shape[0],
    'n_items': movies_only_model['user_item_train'].shape[1],
    'n_train_interactions': movies_only_model['user_item_train'].nnz,
    'books_valid_ndcg@10': np.nan,
    'movies_valid_ndcg@10': movies_only_valid_results['movies']['ndcg@10'],
    'valid_macro_ndcg@10': movies_only_valid_results['movies']['ndcg@10'],
    'books_valid_recall@10': np.nan,
    'movies_valid_recall@10': movies_only_valid_results['movies']['recall@10'],
    'valid_macro_recall@10': movies_only_valid_results['movies']['recall@10'],
    'books_valid_map@5': np.nan,
    'movies_valid_map@5': movies_only_valid_results['movies']['map@5'],
    'valid_macro_map@5': movies_only_valid_results['movies']['map@5'],
    'books_test_ndcg@10': np.nan,
    'movies_test_ndcg@10': movies_only_test_results['movies']['ndcg@10'],
    'test_macro_ndcg@10': movies_only_test_results['movies']['ndcg@10'],
    'books_test_recall@10': np.nan,
    'movies_test_recall@10': movies_only_test_results['movies']['recall@10'],
    'test_macro_recall@10': movies_only_test_results['movies']['recall@10'],
    'books_test_map@5': np.nan,
    'movies_test_map@5': movies_only_test_results['movies']['map@5'],
    'test_macro_map@5': movies_only_test_results['movies']['map@5'],
}

movies_only_result_row


## Baselines: Top-Popular and Random


In [ ]:
from src.evaluation.baselines import make_popularity_recommender, make_random_recommender

popularity_recommend_func = make_popularity_recommender(train_df, domain_item_indices, idx2item)
random_recommend_func = make_random_recommender(domain_item_indices, idx2item, seed=42)

popularity_valid_results, popularity_test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=popularity_recommend_func,
    k=K,
    map_k=5,
)

random_valid_results, random_test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=random_recommend_func,
    k=K,
    map_k=5,
)


def baseline_result_row(model_name, valid_results, test_results):
    return {
        'model': model_name,
        'factors': np.nan,
        'regularization': np.nan,
        'iterations': np.nan,
        'alpha': np.nan,
        'k': K,
        'n_users': n_users,
        'n_items': n_items,
        'n_train_interactions': user_item_train.nnz,
        'books_valid_ndcg@10': valid_results['books']['ndcg@10'],
        'movies_valid_ndcg@10': valid_results['movies']['ndcg@10'],
        'valid_macro_ndcg@10': round((valid_results['books']['ndcg@10'] + valid_results['movies']['ndcg@10']) / 2, 4),
        'books_valid_recall@10': valid_results['books']['recall@10'],
        'movies_valid_recall@10': valid_results['movies']['recall@10'],
        'valid_macro_recall@10': round((valid_results['books']['recall@10'] + valid_results['movies']['recall@10']) / 2, 4),
        'books_valid_map@5': valid_results['books']['map@5'],
        'movies_valid_map@5': valid_results['movies']['map@5'],
        'valid_macro_map@5': round((valid_results['books']['map@5'] + valid_results['movies']['map@5']) / 2, 4),
        'books_test_ndcg@10': test_results['books']['ndcg@10'],
        'movies_test_ndcg@10': test_results['movies']['ndcg@10'],
        'test_macro_ndcg@10': round((test_results['books']['ndcg@10'] + test_results['movies']['ndcg@10']) / 2, 4),
        'books_test_recall@10': test_results['books']['recall@10'],
        'movies_test_recall@10': test_results['movies']['recall@10'],
        'test_macro_recall@10': round((test_results['books']['recall@10'] + test_results['movies']['recall@10']) / 2, 4),
        'books_test_map@5': test_results['books']['map@5'],
        'movies_test_map@5': test_results['movies']['map@5'],
        'test_macro_map@5': round((test_results['books']['map@5'] + test_results['movies']['map@5']) / 2, 4),
    }


popularity_result_row = baseline_result_row('popularity_baseline', popularity_valid_results, popularity_test_results)
random_result_row = baseline_result_row('random_baseline', random_valid_results, random_test_results)


In [ ]:
from src.evaluation.lift import compute_lift_table

print("=== Books (test) ===")
print(compute_lift_table(test_results['books'], popularity_test_results['books'], random_test_results['books']))

print("\n=== Movies (test) ===")
print(compute_lift_table(test_results['movies'], popularity_test_results['movies'], random_test_results['movies']))


In [ ]:
collective_result_row = {
    'model': 'implicit_als_books_movies_joint_weighted',
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': n_users,
    'n_items': n_items,
    'n_train_interactions': user_item_train.nnz,
    'books_valid_ndcg@10': valid_results['books']['ndcg@10'],
    'movies_valid_ndcg@10': valid_results['movies']['ndcg@10'],
    'valid_macro_ndcg@10': round((valid_results['books']['ndcg@10'] + valid_results['movies']['ndcg@10']) / 2, 4),
    'books_valid_recall@10': valid_results['books']['recall@10'],
    'movies_valid_recall@10': valid_results['movies']['recall@10'],
    'valid_macro_recall@10': round((valid_results['books']['recall@10'] + valid_results['movies']['recall@10']) / 2, 4),
    'books_valid_map@5': valid_results['books']['map@5'],
    'movies_valid_map@5': valid_results['movies']['map@5'],
    'valid_macro_map@5': round((valid_results['books']['map@5'] + valid_results['movies']['map@5']) / 2, 4),
    'books_test_ndcg@10': test_results['books']['ndcg@10'],
    'movies_test_ndcg@10': test_results['movies']['ndcg@10'],
    'test_macro_ndcg@10': round((test_results['books']['ndcg@10'] + test_results['movies']['ndcg@10']) / 2, 4),
    'books_test_recall@10': test_results['books']['recall@10'],
    'movies_test_recall@10': test_results['movies']['recall@10'],
    'test_macro_recall@10': round((test_results['books']['recall@10'] + test_results['movies']['recall@10']) / 2, 4),
    'books_test_map@5': test_results['books']['map@5'],
    'movies_test_map@5': test_results['movies']['map@5'],
    'test_macro_map@5': round((test_results['books']['map@5'] + test_results['movies']['map@5']) / 2, 4),
}

als_ablation_results = pd.DataFrame([
    books_only_result_row,
    movies_only_result_row,
    collective_result_row,
    popularity_result_row,
    random_result_row,
])

als_ablation_comparison = als_ablation_results.set_index('model')[[
    'books_test_ndcg@10',
    'books_test_recall@10',
    'books_test_map@5',
    'movies_test_ndcg@10',
    'movies_test_recall@10',
    'movies_test_map@5',
]].rename(index={
    'implicit_als_books_only_weighted': 'Books-only',
    'implicit_als_movies_only_weighted': 'Movies-only',
    'implicit_als_books_movies_joint_weighted': 'Collective',
    'popularity_baseline': 'Top-Popular',
    'random_baseline': 'Random',
}).rename(columns={
    'books_test_ndcg@10': 'Books NDCG@10',
    'books_test_recall@10': 'Books Recall@10',
    'books_test_map@5': 'Books MAP@5',
    'movies_test_ndcg@10': 'Movies NDCG@10',
    'movies_test_recall@10': 'Movies Recall@10',
    'movies_test_map@5': 'Movies MAP@5',
})

os.makedirs('results', exist_ok=True)
als_ablation_comparison.to_csv('results/als_ablation_comparison.csv')
als_ablation_comparison
